In [1]:
import requests
import json
import re
from typing import Dict, List, Tuple, Optional

# API Configuration

In [2]:
OMDB_API_KEY = "3073b2f0"
OMDB_BASE_URL = "http://www.omdbapi.com/"
DADJOKES_BASE_URL = "https://icanhazdadjoke.com/search"

# Movie Data Functions

In [3]:
def get_movie_data(movie_title: str) -> Dict:
    """
    Fetches movie data from the OMDb API.

    Parameters:
    -----------
    movie_title : str
        The title of the movie to search for

    Returns:
    --------
    Dict
        Dictionary containing movie information
    """
    params = {
        "t": movie_title,
        "r": "json",
        "apikey": OMDB_API_KEY
    }

    try:
        response = requests.get(OMDB_BASE_URL, params=params)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f"Error fetching movie data: {e}")
        return {"Response": "False", "Error": "Request failed"}

def rt_rating(movie_data: Dict) -> int:
    """
    Extracts the Rotten Tomatoes rating from movie data.

    Parameters:
    -----------
    movie_data : Dict
        Dictionary containing movie information

    Returns:
    --------
    int
        Rotten Tomatoes rating as integer, or -1 if not found
    """
    ratings = movie_data.get('Ratings', [])
    for rating in ratings:
        if rating['Source'] == 'Rotten Tomatoes':
            return int(rating['Value'].replace("%", ""))
    return -1

# Jokes Functions

In [5]:
def get_joke_data(search_term: str, limit: int = 2) -> Dict:
    """
    Fetches dad jokes containing a specific search term.

    Parameters:
    -----------
    search_term : str
        The term to search for in jokes
    limit : int
        Maximum number of jokes to return (default: 2)

    Returns:
    --------
    Dict
        Dictionary containing joke search results
    """
    params = {
        "term": search_term,
        "limit": limit
    }

    headers = {
        "Accept": "application/json"
    }

    try:
        response = requests.get(DADJOKES_BASE_URL, params=params, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f"Error fetching joke data: {e}")
        return {"results": []}

def get_jokes(plot: str, verbosity: int = 0) -> Tuple[Optional[str], Optional[List[str]]]:
    """
    Finds the longest word in the plot that has associated jokes.

    Parameters:
    -----------
    plot : str
        The movie plot description
    verbosity : int
        If 1, prints which words are being tried

    Returns:
    --------
    Tuple[Optional[str], Optional[List[str]]]
        Tuple of (word_with_jokes, list_of_jokes) or (None, None) if no jokes found
    """
    # Split into words and remove punctuation
    words = plot.split()
    words = [w.strip(",.!;:\"'()[]{}") for w in words]

    # Sort by length (longest first), then alphabetically for ties
    sorted_words = sorted(words, key=lambda word: (-len(word), word.lower()))

    for word in sorted_words:
        if verbosity:
            print(f"Trying word: {word}")

        # Skip very short words and common words that likely won't have jokes
        if len(word) < 4:
            continue

        joke_data = get_joke_data(word)
        if joke_data["results"]:
            jokes = [joke["joke"] for joke in joke_data["results"]]
            return (word, jokes)

    return (None, None)

# Text Formatting Functions

In [6]:
def highlight(word: str, sentence: str) -> str:
    """
    Highlights a word in a sentence by surrounding it with asterisks.

    Parameters:
    -----------
    word : str
        The word to highlight
    sentence : str
        The sentence containing the word

    Returns:
    --------
    str
        The sentence with the word highlighted
    """

    return re.sub(re.escape(word), f"**{word}**", sentence, flags=re.IGNORECASE)


# Main Function

In [7]:
def haha_me(movie_title: str, verbosity: int = 0) -> str:
    """
    Creates a formatted string with movie info and related dad jokes.

    Parameters:
    -----------
    movie_title : str
        The title of the movie
    verbosity : int
        If 1, prints debugging information

    Returns:
    --------
    str
        Formatted string with movie info and jokes
    """
    # Get movie data
    movie_data = get_movie_data(movie_title)

    # Check if movie was found
    if movie_data.get('Response') == 'False':
        return f"No movie found: {movie_title}"

    # Try to find jokes
    joke_result = get_jokes(movie_data['Plot'], verbosity)

    if joke_result == (None, None):
        return "I've got no jokes about this movie. It's too serious!"

    joke_word, jokes = joke_result

    # Get Rotten Tomatoes rating
    rating = rt_rating(movie_data)
    rating_str = f"{rating}%" if rating != -1 else "N/A"

    # Create the output string
    output_lines = []
    output_lines.append(movie_title)
    output_lines.append(f"Rotten Tomatoes rating: {rating_str}")
    output_lines.append(highlight(joke_word, movie_data['Plot']))
    output_lines.append(highlight(joke_word, f"Speaking of {joke_word}, that reminds me of some jokes."))

    # Add rating-based comment
    if rating == -1:
        output_lines.append("Hope you like them!")
    elif rating < 70:
        output_lines.append("Hope they're better than the movie!")
    else:
        output_lines.append("Hope they're as good as the movie!")

    output_lines.append("")  # Empty line

    # Add highlighted jokes
    highlighted_jokes = [highlight(joke_word, joke) for joke in jokes]
    output_lines.extend(highlighted_jokes)

    return "\n".join(output_lines)

# Testing And Demo Functions

In [8]:
def main():
    """Main function to demonstrate the movie jokes mashup."""

    print("🎬 Movie and Jokes Mashup 🎬")
    print("=" * 40)

    # Test with some popular movies
    test_movies = [
        "Black Panther",
        "Back to the Future",
        "The Avengers",
        "Deadpool"
    ]

    for movie in test_movies:
        print(f"Testing: {movie}")
        print("-" * 30)
        result = haha_me(movie, verbosity=1)
        print(result)
        print("\n" + "="*50 + "\n")

def interactive_mode():
    """Interactive mode for user input."""
    print("🎬 Interactive Movie Jokes Generator 🎬")
    print("Enter 'quit' to exit")
    print()


    while True:
        movie_title = input("Enter a movie title: ").strip()

        if movie_title.lower() in ['quit', 'exit', 'q']:
            print("Thanks for using the Movie Jokes Generator! 👋")
            break

        if not movie_title:
            continue

        print("\nGenerating jokes...")
        result = haha_me(movie_title, verbosity=0)
        print("\n" + "="*50)
        print(result)
        print("="*50 + "\n")

In [10]:
main()

🎬 Movie and Jokes Mashup 🎬
Testing: Black Panther
------------------------------
Trying word: challenger
Trying word: country's
Trying word: advanced
Trying word: confront
Trying word: T'Challa
Trying word: forward
Black Panther
Rotten Tomatoes rating: 96%
T'Challa, heir to the hidden but advanced kingdom of Wakanda, must step **forward** to lead his people into a new future and must confront a challenger from his country's past.
Speaking of **forward**, that reminds me of some jokes.
Hope they're as good as the movie!

Sometimes I tuck my knees into my chest and lean **forward**.  That’s just how I roll.
Why do scuba divers fall backwards into the water? Because if they fell **forward**s they’d still be in the boat.


Testing: Back to the Future
------------------------------
Trying word: time-traveling
Trying word: accidentally
Back to the Future
Rotten Tomatoes rating: 93%
Marty McFly, a 17-year-old high school student, is **accidentally** sent 30 years into the past in a time-trave

# **Movie Jokes Generator**

In [11]:
interactive_mode()

🎬 Interactive Movie Jokes Generator 🎬
Enter 'quit' to exit

Enter a movie title: Spider-Man

Generating jokes...

Spider-Man
Rotten Tomatoes rating: 89%
After being bitten by a genetically-modified **spider**, a shy teenager gains **spider**-like abilities that he uses to fight injustice as a masked superhero and face a vengeful enemy.
Speaking of **spider**, that reminds me of some jokes.
Hope they're as good as the movie!

Wife told me to take the **spider** out instead of killing it... We had some drinks, cool guy, wants to be a web developer.

Enter a movie title: Joker

Generating jokes...

Joker
Rotten Tomatoes rating: 68%
Arthur Fleck, a party clown and a failed stand-up **comedian**, leads an impoverished life with his ailing mother. However, when society shuns him and brands him as a freak, he decides to embrace the life of chaos in Gotham City.
Speaking of **comedian**, that reminds me of some jokes.
Hope they're better than the movie!

They laughed when I said I wanted to be